In [1]:
# Cell 1: build the pipeline once
from geoscholar.index import Embedder, Index
from geoscholar.retrieval import Retriever, Reranker
from geoscholar.rag import RAGPipeline, build_prompt

retriever = Retriever(Index(Embedder()), reranker=Reranker())
rag = RAGPipeline(retriever)

def show(question, **kwargs):
    r = RAGPipeline(retriever, **kwargs).answer(question) if kwargs else rag.answer(question)
    print("Q:", question, "\n")
    print(r["answer_display"], "\n")
    for s in r["sources"]:
        mark = "*" if s["n"] in r["cited"] else " "
        print(f" {mark} S{s['n']}  {s['paper_id']}  [{s['section'][:30]}]  {s['title'][:50]}")
    print(f"\nrefused={r['refused']}  invalid={r['invalid_citations']}  {r['timings_ms']}  "
          f"prompt={r['prompt_tokens']} tok  {r['tokens_per_s']} tok/s")
    return r

r = show("What evaluation metrics are used for building change detection?")

/home/jthe/Dev/tutes/geoscholar/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 15726.59it/s]


Q: What evaluation metrics are used for building change detection? 

The evaluation metrics used for building change detection are precision (mPre), recall (mRec), F1 score (mF1), and mean Intersection over Union (mIoU) [2607.27969, 3) Evaluation metrics:]. Additionally, the building change dynamics score (BCDS) is proposed to evaluate predictions from the perspective of semantic preservation in building change dynamics [2607.27969, 3) Evaluation metrics:]. For single-change categories, the BCDS evaluates the prediction quality from three aspects: change existence, single-change attribute, and single-change temporal distance [2607.27969, 3) Evaluation metrics:]. The BCDS is calculated by equally averaging the three scores for single-change categories [2607.27969, 3) Evaluation metrics:]. 

Missing: Information on specific datasets used for building change detection. 

 * S1  2607.27969  [3) Evaluation metrics:]  FootprintNet: State-Transition-Guided Dynamic Foot
   S2  2608.24263  [Dat

In [3]:
# Cell 2: an unanswerable question, and a look at the raw prompt
show("What is the capital of Australia?")
print(build_prompt("test", r["sources"][:2]))

Q: What is the capital of Australia? 

I could not find this in the indexed papers. 

   S1  2608.29793  [3 City3D-MultiGen Benchmark]  GridFlow: Structured Latent Flow for Seamless City
   S2  2608.26471  [I. INTRODUCTION]  Mapping Woody Vegetation from Multi-Source Imagery
   S3  2609.30356  [Most variation within cities r]  AlphaEarth distinguishes cities but compresses urb
   S4  2609.30356  [S7. Weather and construction a]  AlphaEarth distinguishes cities but compresses urb
   S5  2609.30356  [S8.13. Temporal signal at cons]  AlphaEarth distinguishes cities but compresses urb

refused=True  invalid=[]  {'retrieve': 31.4, 'rerank': 358.6, 'generate': 679.6, 'total': 1069.6}  prompt=1058 tok  68.7 tok/s
Sources:

[S1] FootprintNet: State-Transition-Guided Dynamic Footprint Learning for Multi-temporal Remote Sensing Change Detection (2026), section: 3) Evaluation metrics:
We adopt widely used evaluation metrics, including precision (mPre), recall (mRec), F1 score (mF1), and mean Inte

In [5]:
# In 05_answers.ipynb
def check_absent(question, k=8):
    for score, h in retriever.search(question, k=k, mode="hybrid"):
        print(f"{h['paper_id']}  [{h['section'][:25]}]  {h['text'][:110]}")

check_absent("<your candidate question>")

2608.02039  [B.2. Leaf-guided Question]  Question writers select a task and operational leaf before drafting an item. They bind the item to a visible t
2608.02039  [Leaf 22: Relation-constra]  Options: A. From 6 s to 12 s B. From 2 s to 8 s C. From 0 s to 11 s D. From 0 s to 6 s E. From 10 s to 16 s Cl
2607.15942  [Main MLRS prompt.]  Where we insert the ground truth and provided captions in place of {gt}, and {pred} respectively. G-Eval promp
2610.18388  [System prompt for the Dia]  Candidate serialization and output scaffold. Each candidate is serialized using the five fields below. An unpa
2610.18369  [Initial Generation Prompt]  You are a remote sensing visual question answering expert. Answer based solely on what you directly observe in
2610.18388  [Image quality]  Reasoner. The Reasoner receives the image and question, including its answer options and any spatial annotatio
2609.00814  [Main Agent]  You are given: -the candidate the Main Agent selected; -per-class box counts; -imb

In [7]:
# In 05_answers.ipynb
import json
from geoscholar import config

rows = [json.loads(l) for l in open(config.EVAL_DIR / "results" / "answers_baseline.jsonl")]
for r in rows[:10]:
    print("Q:", r["question"])
    print("A:", r["answer_display"])
    print("   label:", r["paper_ids"], " sources:", [s["paper_id"] for s in r["sources"]], "\n")

Q: What machine learning algorithms are used for urban land cover classification in the proposed comparison?
A: The machine learning algorithms used for urban land cover classification in the proposed comparison are:

* Classical ML models: Logistic Regression, Decision Tree, Random Forest, SVM, kNN, Naive Bayes, Gradient Boosting, AdaBoost, XGBoost, CatBoost, and a shallow MLP [2609.19010, 3 Methodology].
* Tabular Deep Learning (TDL) models: TabNet, TabTransformer, FT-Transformer, a 1D CNN, and TabSeq [2609.19010, 3 Methodology].

Note: The source [2609.19010, 1 Introduction] and [2609.19010, Abstract] do not provide information on the specific machine learning algorithms used, but [2609.19010, 3 Methodology] provides the list of algorithms used in the comparison.
   label: ['2609.19010']  sources: ['2609.19010', '2609.19010', '2608.12001', '2608.12001', '2609.19010'] 

Q: What are the improvements in geolocalization accuracy achieved by LoRetta compared to the baseline on different 